# 📋 Excel Files Validator (Enhanced)
## Comprehensive Data Quality & Integrity Check
- Validates all Excel files in input_files folder
- Auto-detects header row (default: Line 2)
- Checks for duplicate files
- Shows date ranges for consistency
- Detects data quality issues

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import for visualization

import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

print('✓ Libraries loaded successfully!')

✓ Libraries loaded successfully!


### Step 1: Configuration - Set Header Row

In [2]:
# 📌 CONFIGURATION - Set your header row here
# Default: 1 (which means Line 2, since counting starts from 0)
# Change this if your files have headers in a different row

HEADER_ROW = 1  # Change this if needed (0 = Line 1, 1 = Line 2, 2 = Line 3, etc.)

print('\n' + '='*80)
print('CONFIGURATION')
print('='*80)
print('\n' + f'📋 Assuming header starts from: LINE {HEADER_ROW + 1} for all files'+'\n')


CONFIGURATION

📋 Assuming header starts from: LINE 2 for all files



### Step 2: Setup & File Discovery

In [3]:
# Define input folder path
INPUT_FOLDER = 'input_files'

# Create folder if it doesn't exist
Path(INPUT_FOLDER).mkdir(exist_ok=True)

# Find all Excel files
excel_files = list(Path(INPUT_FOLDER).glob('*.xlsx')) + list(Path(INPUT_FOLDER).glob('*.xls'))

print('\n' + '='*80)
print('FILE DISCOVERY')
print('='*80)
if not excel_files:
    print('❌ No Excel files found in input_files folder!')
    print('   Please place your Excel files in the input_files folder and run again.')
else:
    print(f'✅ Found {len(excel_files)} file(s):')
    for i, file in enumerate(excel_files, 1):
        file_size_mb = file.stat().st_size / (1024*1024)
        print(f'   {i}. {file.name} ({file_size_mb:.2f} MB)')

excel_files = sorted(excel_files)


FILE DISCOVERY
✅ Found 3 file(s):
   1. Base File Updation North Region-AUG-26.xlsx (0.15 MB)
   2. Billing Report.xlsx (0.05 MB)
   3. Billing Report (1).xlsx (0.16 MB)


### Step 3: Define Critical Columns & Validation Rules

In [4]:
# Critical columns for each file type
CRITICAL_COLUMNS = {
    'Base_File': {
        'INR Sheet': ['Customer Name', 'Updated sale person Name', 'Part Number'],
        'USD Sheet': ['Customer Name', 'Updated sale person Name', 'Part Number']
    },
    'Billing_Report': {
        'Line': ['Customer Name', 'Sales Employee Name', 'Total Value'],
        'MOUSER': ['Customer Name', 'Sales Employee Name', 'Total Value'],
        'USD': ['Customer Name', 'Sales Employee', 'Row Total in USD'],
        'Direct': ['Customer/Supplier Name', 'Sales Person']
    }
}

# Validation rules
VALIDATION_RULES = {
    'numeric': ['Total Value', 'Row Total in USD', 'Basic Value', 'Quantity'],
    'date': ['Inv Date', 'Posting Date', 'Document Date'],
    'non_empty': ['Customer Name', 'Sales Employee Name', 'Sales Person']
}

print('✓ Validation rules defined')

✓ Validation rules defined


### Step 4: Enhanced Validation Engine

In [5]:
class FileValidator:
    def __init__(self, file_path, header_row=1):
        self.file_path = file_path
        self.file_name = file_path.name
        self.header_row = header_row
        self.issues = []
        self.warnings = []
        self.info = []
        self.duplicate_rows = []
        self.sheets_data = {}
        self.date_range = None
        self.file_hash = None
        self.date_distribution = None
        
    def validate(self):
        '''Main validation method'''
        try:
            self._check_file_integrity()
            self._load_and_check_sheets()
            self._validate_critical_columns()
            self._validate_data_quality()
            self._extract_date_range()
            self._extract_date_distribution()
            self._calculate_file_hash()
            return True
        except Exception as e:
            self.issues.append(f'Critical Error: {str(e)}')
            return False
    
    def _check_file_integrity(self):
        '''Check if file exists and is readable'''
        if not self.file_path.exists():
            self.issues.append('File does not exist')
            return False
        
        try:
            pd.ExcelFile(self.file_path)
            self.info.append('✓ File is readable')
            return True
        except Exception as e:
            self.issues.append(f'File is corrupted: {str(e)[:50]}')
            return False
    
    def _load_and_check_sheets(self):
        '''Load and validate sheet structure'''
        try:
            xls = pd.ExcelFile(self.file_path)
            sheet_names = xls.sheet_names
            
            if not sheet_names:
                self.issues.append('File has no sheets')
                return
            
            self.info.append(f'✓ Found {len(sheet_names)} sheet(s): {", ".join(sheet_names)}')
            
            for sheet_name in sheet_names:
                try:
                    df = pd.read_excel(self.file_path, sheet_name=sheet_name, header=self.header_row)
                    self.sheets_data[sheet_name] = df
                    
                    if len(df) == 0:
                        self.warnings.append(f"Sheet '{sheet_name}': Empty sheet (0 rows)")
                    else:
                        self.info.append(f"  • '{sheet_name}': {len(df)} rows × {len(df.columns)} columns")
                except Exception as e:
                    self.issues.append(f"Sheet '{sheet_name}': Cannot read - {str(e)[:40]}")
        except Exception as e:
            self.issues.append(f'Cannot read sheets: {str(e)[:50]}')
    
    def _validate_critical_columns(self):
        '''Validate presence of critical columns'''
        for sheet_name, df in self.sheets_data.items():
            if df.empty:
                continue
            
            critical_cols = []
            for file_type, sheets in CRITICAL_COLUMNS.items():
                if file_type.lower() in self.file_name.lower():
                    if sheet_name in sheets:
                        critical_cols = sheets[sheet_name]
                        break
            
            if critical_cols:
                df_columns = [str(col).strip() for col in df.columns]
                
                missing_cols = [col for col in critical_cols if col not in df_columns]
                if missing_cols:
                    self.issues.append(f"Sheet '{sheet_name}': Missing critical columns: {', '.join(missing_cols)}")
                else:
                    self.info.append(f"  ✓ '{sheet_name}': All critical columns present")
    
    def _validate_data_quality(self):
        '''Validate data quality issues - user friendly format'''
        for sheet_name, df in self.sheets_data.items():
            if df.empty:
                continue
            
            # Check for duplicate rows
            duplicates = df.duplicated().sum()
            if duplicates > 0:
                self.duplicate_rows.append((sheet_name, int(duplicates), len(df)))
            
            # Check for completely empty columns
            empty_cols = df.columns[df.isnull().all()].tolist()
            if empty_cols:
                for col in empty_cols:
                    self.warnings.append(f"Sheet '{sheet_name}': Column '{col}' is COMPLETELY EMPTY (has no data at all)")
            
            # Check for columns with >50% missing values
            null_percentages = (df.isnull().sum() / len(df)) * 100
            high_null_cols = null_percentages[null_percentages >= 50].index.tolist()
            if high_null_cols:
                for col in high_null_cols:
                    missing_pct = null_percentages[col]
                    missing_count = df[col].isnull().sum()
                    total_rows = len(df)
                    self.warnings.append(
                        f"Sheet '{sheet_name}': Column '{col}' has {missing_count} missing values out of {total_rows} rows ({missing_pct:.1f}% empty)"
                    )
    
    @staticmethod
    def _is_date_column(col):
        '''Check if column is the BILLING date (Inv Date / Posting Date).
        Customer PO Date is excluded - that is when the customer ordered,
        not when the bill was raised.'''
        col_lower = str(col).lower().strip()
        if 'po date' in col_lower or 'order date' in col_lower:
            return False
        return ('inv date' in col_lower or 'invoice date' in col_lower or
                'posting date' in col_lower or 'document date' in col_lower or
                col_lower == 'date')
    
    def _get_valid_dates(self, df):
        '''Collect valid dates (year >= 2000) from real date columns only'''
        collected = []
        for col in df.columns:
            if self._is_date_column(col):
                try:
                    dates = pd.to_datetime(df[col], errors='coerce').dropna()
                    dates = dates[dates.dt.year >= 2000]  # ignore invalid epoch dates
                    if len(dates) > 0:
                        collected.extend(dates.tolist())
                except:
                    pass
        return collected
    
    def _extract_date_range(self):
        '''Extract date range from data (real date columns only, valid dates only)'''
        all_dates = []
        for sheet_name, df in self.sheets_data.items():
            if df.empty:
                continue
            all_dates.extend(self._get_valid_dates(df))
        
        if all_dates:
            self.date_range = (min(all_dates), max(all_dates))
    
    def _extract_date_distribution(self):
        '''Extract date distribution for visualization'''
        all_dates_list = []
        
        for sheet_name, df in self.sheets_data.items():
            if df.empty:
                continue
            all_dates_list.extend(self._get_valid_dates(df))
        
        if all_dates_list:
            # Create dataframe with dates
            dates_df = pd.DataFrame({'date': all_dates_list})
            # Group by week
            dates_df['week'] = dates_df['date'].dt.to_period('W')
            self.date_distribution = dates_df.groupby('week').size().reset_index(name='count')
            self.date_distribution['week'] = self.date_distribution['week'].dt.start_time
    
    def _calculate_file_hash(self):
        '''Calculate file hash for duplicate detection'''
        import hashlib
        with open(self.file_path, 'rb') as f:
            self.file_hash = hashlib.md5(f.read()).hexdigest()
    
    def print_report(self):
        '''Print validation report'''
        print('\n' + '='*80)
        print(f'FILE: {self.file_name}')
        print('='*80)
        
        if self.issues:
            print('\n🔴 CRITICAL ISSUES (Must be fixed):')
            for issue in self.issues:
                print(f'   ❌ {issue}')
        
        if self.info:
            print('\n🟢 VALID (Good - No problems here):')
            for info in self.info:
                print(f'   {info}')
        
        if self.duplicate_rows:
            print('\n🔁 DUPLICATE ROWS (exact same data appears more than once):')
            for sheet_name, dup_count, total_rows in self.duplicate_rows:
                print(f"   ⚠️  Sheet '{sheet_name}'  →  {dup_count} duplicate rows  (out of {total_rows} rows)")
        
        if self.warnings:
            print('\n🟡 WARNINGS (Review these):')
            for warning in self.warnings:
                print(f'   ⚠️  {warning}')
        
        if self.date_range:
            print('\n📅 DATA DATE RANGE:')
            min_d = self.date_range[0].strftime('%Y-%m-%d')
            max_d = self.date_range[1].strftime('%Y-%m-%d')
            print(f'   From: {min_d}')
            print(f'   To:   {max_d}')
        
        if not self.issues and not self.warnings:
            print('\n✅ NO ISSUES FOUND - File is valid and ready to use!')
        
        print(f'\nSummary: {len(self.issues)} Critical | {len(self.warnings) + len(self.duplicate_rows)} Warnings | {len(self.info)} Valid')

print('✓ Validator class ready')

✓ Validator class ready


### Step 5: Run Validation on All Files

In [6]:
if excel_files:
    print('\n' + '='*80)
    print('VALIDATION PROCESS STARTED')
    print(f'Using header row: LINE {HEADER_ROW + 1}')
    print('='*80)
    
    validation_results = []
    
    for file_path in excel_files:
        validator = FileValidator(file_path, header_row=HEADER_ROW)
        validator.validate()
        validator.print_report()
        validation_results.append(validator)
    
    print('\n' + '='*80)
    print('VALIDATION COMPLETE')
    print('='*80)


VALIDATION PROCESS STARTED
Using header row: LINE 2

FILE: Base File Updation North Region-AUG-26.xlsx

🟢 VALID (Good - No problems here):
   ✓ File is readable
   ✓ Found 2 sheet(s): INR Sheet , USD Sheet 
     • 'INR Sheet ': 1000 rows × 13 columns
     • 'USD Sheet ': 658 rows × 13 columns

🔁 DUPLICATE ROWS (exact same data appears more than once):
   ⚠️  Sheet 'INR Sheet '  →  2 duplicate rows  (out of 1000 rows)
   ⚠️  Sheet 'USD Sheet '  →  14 duplicate rows  (out of 658 rows)

🟡 WARNINGS (Review these):
   ⚠️  Sheet 'INR Sheet ': Column 'Aug'26(B) Qty' has 589 missing values out of 1000 rows (58.9% empty)
   ⚠️  Sheet 'INR Sheet ': Column 'Aug(F) Qty' has 648 missing values out of 1000 rows (64.8% empty)
   ⚠️  Sheet 'INR Sheet ': Column 'Week 1' has 984 missing values out of 1000 rows (98.4% empty)
   ⚠️  Sheet 'INR Sheet ': Column 'Week 2' has 865 missing values out of 1000 rows (86.5% empty)
   ⚠️  Sheet 'INR Sheet ': Column 'Week 3' has 909 missing values out of 1000 rows (

### Step 5B: Duplicate Rows - All Files

In [7]:
# 🔁 ALL DUPLICATE ROWS - ACROSS ALL FILES (easy to spot in one place)
if excel_files and len(validation_results) > 0:
    print('\n' + '='*80)
    print('🔁 DUPLICATE ROWS - ALL FILES')
    print('='*80)

    any_dups = False
    for validator in validation_results:
        if validator.duplicate_rows:
            any_dups = True
            print(f'\n📄 {validator.file_name}')
            for sheet_name, dup_count, total_rows in validator.duplicate_rows:
                pct = (dup_count / total_rows * 100) if total_rows else 0
                print(f"   ⚠️  Sheet '{sheet_name}'  →  {dup_count} duplicate rows  (out of {total_rows} rows, {pct:.1f}%)")

    if not any_dups:
        print('\n✅ No duplicate rows found in any file')
    else:
        print('\n💡 What to do: open the sheet in Excel → Data → Remove Duplicates,')
        print('   or confirm the repeats are genuine before using the file.')


🔁 DUPLICATE ROWS - ALL FILES

📄 Base File Updation North Region-AUG-26.xlsx
   ⚠️  Sheet 'INR Sheet '  →  2 duplicate rows  (out of 1000 rows, 0.2%)
   ⚠️  Sheet 'USD Sheet '  →  14 duplicate rows  (out of 658 rows, 2.1%)

📄 Billing Report (1).xlsx
   ⚠️  Sheet 'USD'  →  6 duplicate rows  (out of 131 rows, 4.6%)

📄 Billing Report.xlsx
   ⚠️  Sheet 'USD'  →  1 duplicate rows  (out of 69 rows, 1.4%)

💡 What to do: open the sheet in Excel → Data → Remove Duplicates,
   or confirm the repeats are genuine before using the file.


### Step 6: Check for Duplicate Files

In [8]:
if excel_files and len(validation_results) > 1:
    print('\n' + '='*80)
    print('DUPLICATE FILE CHECK')
    print('='*80)
    
    found_duplicates = False
    
    for i in range(len(validation_results)):
        for j in range(i+1, len(validation_results)):
            if validation_results[i].file_hash == validation_results[j].file_hash:
                print(f'\n🔴 DUPLICATE FILES FOUND:')
                print(f'   File 1: {validation_results[i].file_name}')
                print(f'   File 2: {validation_results[j].file_name}')
                print(f'   Status: ⚠️  These files are IDENTICAL')
                print(f'   Action: Delete one of them')
                found_duplicates = True
    
    if not found_duplicates:
        print('\n✅ No duplicate files found - All files are unique')


DUPLICATE FILE CHECK

✅ No duplicate files found - All files are unique


### Step 7: Compare Date Ranges

In [9]:
if excel_files and len(validation_results) > 0:
    print('\n' + '='*80)
    print('DATA DATE ANALYSIS')
    print('='*80)
    
    files_with_dates = [v for v in validation_results if v.date_range]
    
    if files_with_dates:
        print('\n📅 DATE SUMMARY BY FILE:')
        print('-'*80)
        
        for validator in files_with_dates:
            min_d = validator.date_range[0]
            max_d = validator.date_range[1]
            days_span = (max_d - min_d).days
            
            print(f'\n📄 {validator.file_name}:')
            print(f'   From: {min_d.strftime("%Y-%m-%d")} → To: {max_d.strftime("%Y-%m-%d")} ({days_span} days)')
        
        # Overall date range check
        all_min = min([v.date_range[0] for v in files_with_dates])
        all_max = max([v.date_range[1] for v in files_with_dates])
        
        print(f'\n📊 OVERALL DATE RANGE: {all_min.strftime("%Y-%m-%d")} to {all_max.strftime("%Y-%m-%d")}')
        
        # Check consistency
        if len(files_with_dates) > 1:
            months = set()
            for v in files_with_dates:
                months.add((v.date_range[1].year, v.date_range[1].month))
            if len(months) == 1:
                print(f'   ✅ All files have their latest data in the SAME MONTH - Good!')
            else:
                print(f'   ⚠️  Files end in DIFFERENT months - check if intentional')


DATA DATE ANALYSIS

📅 DATE SUMMARY BY FILE:
--------------------------------------------------------------------------------

📄 Billing Report (1).xlsx:
   From: 2026-07-20 → To: 2026-08-14 (25 days)

📄 Billing Report.xlsx:
   From: 2026-07-20 → To: 2026-08-07 (18 days)

📊 OVERALL DATE RANGE: 2026-07-20 to 2026-08-14
   ✅ All files have their latest data in the SAME MONTH - Good!


### Step 7B: Week-Wise Data Distribution Graph

In [10]:
# Week-Wise Distribution Graph (files without dates are ignored automatically)
if excel_files and len(validation_results) > 0:
    files_with_valid_dates = [v for v in validation_results 
                              if v.date_distribution is not None and len(v.date_distribution) > 0]
    
    if files_with_valid_dates:
        print('\n' + '='*80)
        print('📊 WEEK-WISE DATA DISTRIBUTION')
        print('='*80)
        
        # Text preview first
        for validator in files_with_valid_dates:
            dist = validator.date_distribution
            total = int(dist['count'].sum())
            min_d = validator.date_range[0].strftime('%Y-%m-%d')
            max_d = validator.date_range[1].strftime('%Y-%m-%d')
            
            print(f'\n📄 {validator.file_name}:')
            print(f'   Date Range: {min_d} to {max_d}')
            print(f'   Total Records: {total}')
            print(f'   Week-Wise Breakdown:')
            for _, row in dist.iterrows():
                week_date = row['week'].strftime('%Y-%m-%d')
                count = int(row['count'])
                bar = '█' * max(1, count // 20)
                print(f'      Week of {week_date}: {count:5d} records  {bar}')
        
        # Now the graphs
        num_files = len(files_with_valid_dates)
        
        if num_files == 1:
            axes = [axes]



📊 WEEK-WISE DATA DISTRIBUTION

📄 Billing Report (1).xlsx:
   Date Range: 2026-07-20 to 2026-08-14
   Total Records: 800
   Week-Wise Breakdown:
      Week of 2026-07-20:    23 records  █
      Week of 2026-07-27:    29 records  █
      Week of 2026-08-03:   308 records  ███████████████
      Week of 2026-08-10:   440 records  ██████████████████████

📄 Billing Report.xlsx:
   Date Range: 2026-07-20 to 2026-08-07
   Total Records: 186
   Week-Wise Breakdown:
      Week of 2026-07-20:    20 records  █
      Week of 2026-07-27:    25 records  █
      Week of 2026-08-03:   141 records  ███████


In [11]:
if excel_files:
    print('\n' + '='*80)
    print('OVERALL VALIDATION SUMMARY')
    print('='*80)
    
    total_issues = sum(len(v.issues) for v in validation_results)
    total_warnings = sum(len(v.warnings) + len(v.duplicate_rows) for v in validation_results)
    files_with_errors = sum(1 for v in validation_results if v.issues)
    files_passed = len(validation_results) - files_with_errors
    
    print(f'\n📊 Statistics:')
    print(f'   Total Files Analyzed: {len(validation_results)}')
    print(f'   Files Passed ✅: {files_passed}')
    print(f'   Files with Issues ❌: {files_with_errors}')
    print(f'   Total Critical Issues: {total_issues}')
    print(f'   Total Warnings: {total_warnings}')
    
    print(f'\n📋 Files Status:')
    for v in validation_results:
        status = '✅ PASS' if not v.issues else '❌ FAIL'
        print(f'   {v.file_name:45} {status} ({len(v.issues)} issues, {len(v.warnings) + len(v.duplicate_rows)} warnings)')
    
    print(f'\n💡 Recommendations:')
    
    if files_passed == len(validation_results):
        print(f'   ✅ All files are valid and ready for processing!')
    else:
        print(f'   ⚠️  Review CRITICAL ISSUES above for each file')
        print(f'   ⚠️  Fix issues before using files in analysis')
    
    print('\n' + '='*80)


OVERALL VALIDATION SUMMARY

📊 Statistics:
   Total Files Analyzed: 3
   Files Passed ✅: 3
   Files with Issues ❌: 0
   Total Critical Issues: 0
   Total Warnings: 25

📋 Files Status:
   Base File Updation North Region-AUG-26.xlsx   ✅ PASS (0 issues, 14 warnings)
   Billing Report (1).xlsx                       ✅ PASS (0 issues, 6 warnings)
   Billing Report.xlsx                           ✅ PASS (0 issues, 5 warnings)

💡 Recommendations:
   ✅ All files are valid and ready for processing!

